In [20]:
## Part 1: Set Up and Generate Multi-Layout Data (10 min)
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time, os, random

spark = SparkSession.builder \
    .appName("StreamPulse-PartitionAudit") \
    .master("local[4]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()


In [21]:
## Generate 800K listening events:
random.seed(42)
data = []
for i in range(800000):
    data.append((
        f"EVT-{i+1:07d}",
        f"USR-{random.randint(1, 100000):06d}",
        random.choice(["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic", "R&B"]),
        random.choice(["mobile", "desktop", "smart_speaker", "tablet"]),
        random.randint(15, 350),
        random.choice([True, False]),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))

df = spark.createDataFrame(data,
    ["event_id", "user_id", "genre", "device", "duration_sec", "completed", "event_date"]) \
    .withColumn("event_date", col("event_date").cast("date")) \
    .withColumn("month", month(col("event_date")))


In [22]:
## Save in multiple file layouts:
for n in [1, 4, 8, 20, 100]:
    output = f"audit_data/layout_{n}"
    if n <= df.rdd.getNumPartitions():
        df.coalesce(n).write.parquet(output, mode="overwrite")
    else:
        df.repartition(n).write.parquet(output, mode="overwrite")
    file_count = len([f for f in os.listdir(output) if f.endswith(".parquet")])
    print(f"Layout {n:>3}: {file_count} files")


Layout   1: 1 files
Layout   4: 4 files
Layout   8: 8 files
Layout  20: 20 files
Layout 100: 100 files


In [23]:
## Part 2: Input Partition Exploration (15 min)
## For each layout, measure:
# Partition count after read
# Row distribution across partitions
# Read + groupBy time

def partition_profile(path, label):
    df = spark.read.parquet(path)
    num_parts = df.rdd.getNumPartitions()

    dist = df.withColumn("pid", spark_partition_id()) \
        .groupBy("pid").agg(count("*").alias("rows")).toPandas()

    min_rows = dist["rows"].min()
    max_rows = dist["rows"].max()
    avg_rows = dist["rows"].mean()
    ratio = max_rows / min_rows if min_rows > 0 else float("inf")

    start = time.time()
    df.groupBy("genre").agg(sum("duration_sec"), count("*")).collect()
    elapsed = time.time() - start

    print(f"{label:<20} parts={num_parts:<4} min={min_rows:<8.0f} max={max_rows:<8.0f} "
          f"ratio={ratio:<5.1f} groupBy={elapsed:.3f}s")
    return num_parts, elapsed

print(f"{'Layout':<20} {'Parts':<6} {'Min Rows':<10} {'Max Rows':<10} {'Ratio':<7} {'GroupBy'}")
print("-" * 75)
for n in [1, 4, 8, 20, 100]:
    partition_profile(f"audit_data/layout_{n}", f"Layout {n} files")


Layout               Parts  Min Rows   Max Rows   Ratio   GroupBy
---------------------------------------------------------------------------
Layout 1 files       parts=3    min=800000   max=800000   ratio=1.0   groupBy=2.840s
Layout 4 files       parts=4    min=199680   max=200704   ratio=1.0   groupBy=2.376s
Layout 8 files       parts=4    min=200000   max=200000   ratio=1.0   groupBy=1.694s
Layout 20 files      parts=4    min=200000   max=200000   ratio=1.0   groupBy=1.978s
Layout 100 files     parts=4    min=199984   max=200012   ratio=1.0   groupBy=1.503s


In [ ]:
## Questions to answer:

# Which layout gave the best groupBy performance on 4 cores? - layout with 100 files
# What's the ideal input partition count for a 4-core machine? - partition count 4
# Did the 100-file layout perform worse than 20? Why? - NO, it did not perfome worse than the 20 file layout

In [24]:
## Part 3: Shuffle Partition Tuning (15 min)
## Test different spark.sql.shuffle.partitions settings:

df = spark.read.parquet("audit_data/layout_8")

print(f"{'Shuffle Parts':<15} {'GroupBy Time':<13} {'Join Time':<13}")
print("-" * 45)

lookup = spark.createDataFrame(
    [("Pop", 1), ("Rock", 2), ("Hip-Hop", 3), ("Jazz", 4), ("Electronic", 5), ("R&B", 6)],
    ["genre", "genre_id"])

for n in [2, 4, 8, 16, 50, 200, 1000]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n))

    start = time.time()
    df.groupBy("genre", "device").agg(sum("duration_sec"), count("*")).collect()
    t_group = time.time() - start

    start = time.time()
    df.join(lookup, "genre").groupBy("genre_id").agg(count("*")).collect()
    t_join = time.time() - start

    print(f"  {n:>5}         {t_group:.3f}s        {t_join:.3f}s")

spark.conf.set("spark.sql.shuffle.partitions", "8")


Shuffle Parts   GroupBy Time  Join Time    
---------------------------------------------
      2         1.025s        2.359s
      4         0.414s        1.875s
      8         0.550s        2.498s
     16         0.475s        1.548s
     50         0.780s        1.749s
    200         1.914s        2.368s
   1000         7.275s        7.379s


In [ ]:
## Optimal shuffle partition count is 16 with a Groupby time of 0.475s and a join time of 1.548s

In [25]:
## Part 4: Track Partitions Through Pipeline Stages (10 min)
## Build a pipeline and track partition count at each step:

df = spark.read.parquet("audit_data/layout_8")

stages = {}
stages["1. Read"] = df.rdd.getNumPartitions()

df_filtered = df.filter(col("completed") == True)
stages["2. Filter"] = df_filtered.rdd.getNumPartitions()

df_selected = df_filtered.select("event_id", "genre", "device", "duration_sec")
stages["3. Select"] = df_selected.rdd.getNumPartitions()

df_grouped = df_filtered.groupBy("genre").agg(count("*"))
stages["4. GroupBy"] = df_grouped.rdd.getNumPartitions()

df_sorted = df_grouped.orderBy(col("count(1)").desc())
stages["5. OrderBy"] = df_sorted.rdd.getNumPartitions()

df_coalesced = df_filtered.coalesce(4)
stages["6. Coalesce(4)"] = df_coalesced.rdd.getNumPartitions()

df_repartitioned = df_filtered.repartition(16)
stages["7. Repartition(16)"] = df_repartitioned.rdd.getNumPartitions()

print(f"{'Stage':<25} {'Partitions':>12} {'Change'}")
print("-" * 55)
prev = None
for stage, parts in stages.items():
    change = ""
    if prev is not None:
        if parts > prev:
            change = f"↑ increased from {prev}"
        elif parts < prev:
            change = f"↓ decreased from {prev}"
        else:
            change = "= unchanged"
    print(f"  {stage:<23} {parts:>10}   {change}")
    prev = parts


Stage                       Partitions Change
-------------------------------------------------------
  1. Read                          4   
  2. Filter                        4   = unchanged
  3. Select                        4   = unchanged
  4. GroupBy                       8   ↑ increased from 4
  5. OrderBy                       6   ↓ decreased from 8
  6. Coalesce(4)                   4   ↓ decreased from 6
  7. Repartition(16)              16   ↑ increased from 4


In [27]:
## Part 5: Output File Control (10 min)
## Experiment with file count at write time:

df = spark.read.parquet("audit_data/layout_8")
df_processed = df.filter(col("completed") == True) \
    .groupBy("genre", "month") \
    .agg(sum("duration_sec").alias("total_duration"), count("*").alias("plays"))

for n in [1, 4, 8, 20]:
    output = f"audit_data/output_{n}"
    if n <= df_processed.rdd.getNumPartitions():
        df_processed.coalesce(n).write.parquet(output, mode="overwrite")
    else:
        df_processed.repartition(n).write.parquet(output, mode="overwrite")

    files = [f for f in os.listdir(output) if f.endswith(".parquet")]
    total_size = __builtins__.sum(os.path.getsize(os.path.join(output, f)) for f in files)
    avg_size = total_size / len(files) if files else 0

    print(f"  {n:>2} partitions → {len(files)} files, "
          f"total {total_size/1024:.0f} KB, avg {avg_size/1024:.1f} KB/file")

   1 partitions → 1 files, total 2 KB, avg 2.2 KB/file
   4 partitions → 4 files, total 6 KB, avg 1.6 KB/file
   8 partitions → 8 files, total 12 KB, avg 1.5 KB/file
  20 partitions → 20 files, total 27 KB, avg 1.3 KB/file


In [ ]:
## Part 6: Write Partition Strategy Document (15 min)
## Based on your experiments, create a StreamPulse Partition Standards document:

# StreamPulse Partition Standards

## Environment: Local Development (4 cores)

### Input Partitions
- Target: 4 partitions (based on Part 2 findings)
- Match to: 100 x core count

### Shuffle Partitions
- spark.sql.shuffle.partitions: 16 (based on Part 3 findings)
- Reasoning:has the most optimal shuffle partition count ans fastest in performance

### Output Partitions
- Target file size: 20
- Use coalesce(4) before write: 4 partitions
- Use partitionBy for: < 100 columns

### Production Cluster Recommendations
- Shuffle partitions: 2-3x total executor cores
- Target partition size: 128MB - 256MB
- Maximum partition count: 10,000
